# Phase 2 — Temperature & Pressure Loading

**Well**: 15/9-F-1 A · **Field**: Volve, Norwegian North Sea

This notebook derives the three computed logs required by later phases:

| Output curve | Used in |
|---|---|
| `TEMP` — formation temperature (°C) | Batzle-Wang fluid properties (Ph. 12), Rw correction |
| `PORE_PRESS` — hydrostatic pore pressure (MPa) | Pressure log, RPM (Ph. 11) |
| `DIFF_PRESS` — differential / effective pressure (MPa) | Stress-sensitive RPM (Ph. 11) |

**Key assumptions (can be revisited)**:
- No raw BHT log in the LAS file — Horner method is demonstrated as a worked example using representative BHT values; the final gradient matches published Volve field data
- Volve Hugin Fm formation water salinity: **57,500 ppm NaCl** (representative; refine from produced water samples if available)
- Reservoir is **normally pressured** — hydrostatic gradient used for pore pressure
- Overburden above the logging interval derived from a compaction trend; measured RHOB used within the logging interval

In [ ]:
import lasio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.stats import linregress
from pathlib import Path

# ── Paths ───────────────────────────────────────────────────────────────────
WELL_FILE = Path('../wells/15_9-F-1A.LAS')
TOPS_FILE = Path('../wells/Volve_formation_tops.csv')
OUT_FILE  = Path('../wells/15_9-F-1A_computed.parquet')   # output for later notebooks

# ── Well geometry — Volve field, 15/9-F-1 A ─────────────────────────────────
# Derived from formation tops file: KB_above_MSL = TVD_top - |TVDSS_top|
KB_ABOVE_MSL  = 54.9     # m  (Kelly Bushing elevation above Mean Sea Level)
WATER_DEPTH   = 91.1     # m  (seabed depth below MSL)

# ── Temperature — Volve published geothermal gradient ───────────────────────
T_SEABED      = 5.0      # °C  (seabed / bottom-water temperature)
GEOTHERM      = 0.031    # °C/m (gradient from seabed; ~31°C/km)

# ── Formation water — Volve Hugin Fm ────────────────────────────────────────
SALINITY_PPM  = 57_500   # ppm NaCl equivalent

# ── Pressure ─────────────────────────────────────────────────────────────────
RHO_SEAWATER  = 1.025    # g/cc
G_MPA         = 9.81e-3  # MPa per (g/cc · m)  [= 9810 Pa / (1000 kg/m³ · m)]

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family' : 'DejaVu Sans',
    'font.size'   : 9,
    'axes.linewidth': 0.8,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'figure.dpi'  : 120,
})

## Step 2.1 — Load Data & Build MD → TVD Conversion

15/9-F-1 A is a deviated well. Depths in the LAS file are **measured depth (MD)**; temperature and pressure calculations require **TVDss** (True Vertical Depth below sea level).

We build a MD → TVDss lookup from the formation tops file (12 tie-points across the logging interval) and linearly interpolate to every 0.1 m depth sample.

In [ ]:
# ── Load LAS ─────────────────────────────────────────────────────────────────
las = lasio.read(WELL_FILE)
df  = las.df()
df.replace(-999.25, np.nan, inplace=True)
df.index.name = 'DEPTH_MD'

# ── Formation tops — depth tie-points ────────────────────────────────────────
all_tops = pd.read_csv(TOPS_FILE)
f1a = (
    all_tops[all_tops['WELL'] == 'NO 15/9-F-1 A']
    .sort_values('DEPTH')
    .drop_duplicates(subset='DEPTH')   # remove duplicate MD entries
    .reset_index(drop=True)
)

# TVDSS convention in file: negative = below MSL (elevation convention)
# Convert to positive-downward for calculations
f1a['TVDSS_ABS'] = f1a['TVDSS'].abs()

# ── Interpolate to every LAS depth sample ─────────────────────────────────────
md = df.index.values

df['TVD']      = np.interp(md, f1a['DEPTH'].values, f1a['TVD'].values)
df['TVDSS_ABS'] = np.interp(md, f1a['DEPTH'].values, f1a['TVDSS_ABS'].values)
df['DEPTH_BELOW_SEABED'] = np.maximum(df['TVDSS_ABS'] - WATER_DEPTH, 0.0)

# ── Quick sanity check ────────────────────────────────────────────────────────
print(f"KB above MSL          : {KB_ABOVE_MSL:.1f} m")
print(f"Seabed depth (MSL)    : {WATER_DEPTH:.1f} m")
print(f"MD range              : {md.min():.0f}–{md.max():.0f} m")
print(f"TVD range             : {df['TVD'].min():.0f}–{df['TVD'].max():.0f} m")
print(f"TVDss range (abs)     : {df['TVDSS_ABS'].min():.0f}–{df['TVDSS_ABS'].max():.0f} m")
print()
print("Formation tops used for MD→TVD conversion:")
print(f"  {'Formation':<35} {'MD':>8} {'TVD':>8} {'TVDss':>8}")
print("  " + "─"*62)
for _, r in f1a.iterrows():
    print(f"  {r['PICKS']:<35} {r['DEPTH']:>8.1f} {r['TVD']:>8.1f} {-r['TVDSS_ABS']:>8.1f}")


## Step 2.2 — Temperature: Horner Correction Method

Wireline tools are run shortly after the borehole has been circulated with cold drilling mud. The measured Bottom Hole Temperature (BHT) is therefore lower than the true static formation temperature.

The **Horner correction** uses repeated temperature measurements at increasing shut-in times to extrapolate back to the undisturbed formation temperature:

$$T_{\text{meas}} = T_{\text{static}} - k \cdot \log_{10}\!\left(\frac{t_c + \Delta t}{\Delta t}\right)$$

where $t_c$ is the total circulation time and $\Delta t$ is the time since circulation stopped. Plotting $T_{\text{meas}}$ vs the Horner ratio gives a straight line; extrapolating to ratio = 0 (infinite shut-in) yields $T_{\text{static}}$.

The example below uses **representative BHT values** for a well of this depth and gradient — no raw BHT data is stored in the LAS header.

In [ ]:
# ── Representative BHT measurements at ~3100 m TVDss ─────────────────────────
# Four surveys taken at increasing shut-in times after the same circulation event
bht = pd.DataFrame({
    'survey'    : ['Survey 1', 'Survey 2', 'Survey 3', 'Survey 4'],
    't_circ_h'  : [10,   10,   10,   10  ],   # hours of circulation before surveys
    'delta_t_h' : [ 2,    4,    8,   14  ],   # hours since circulation stopped
    'bht_c'     : [74,   81,   87,   92  ],   # measured BHT (°C)
})
bht['horner_ratio'] = np.log10(
    (bht['t_circ_h'] + bht['delta_t_h']) / bht['delta_t_h']
)

# ── Linear regression through the Horner plot ─────────────────────────────────
slope, intercept, r_val, _, _ = linregress(bht['horner_ratio'], bht['bht_c'])
T_STATIC_HORNER = intercept   # extrapolated static temperature at ratio = 0

print("Horner correction results (at ~3100 m TVDss):")
print(f"  {'Survey':<12} {'t_circ (h)':>12} {'Δt (h)':>8} {'BHT (°C)':>10} {'Horner ratio':>14}")
print("  " + "─"*60)
for _, r in bht.iterrows():
    print(f"  {r['survey']:<12} {r['t_circ_h']:>12.0f} {r['delta_t_h']:>8.0f} "
          f"{r['bht_c']:>10.0f} {r['horner_ratio']:>14.3f}")
print()
print(f"  Regression slope     : {slope:.2f} °C / Horner unit")
print(f"  Extrapolated T_static: {T_STATIC_HORNER:.1f} °C  (at Horner ratio = 0)")
print(f"  R²                   : {r_val**2:.4f}")

In [ ]:
# ── Horner Plot ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))

# Scatter points
ax.scatter(bht['horner_ratio'], bht['bht_c'],
           s=70, color='#2980B9', zorder=5, label='BHT surveys')
for _, r in bht.iterrows():
    ax.annotate(f"{r['survey']}\nΔt={r['delta_t_h']:.0f}h",
                xy=(r['horner_ratio'], r['bht_c']),
                xytext=(6, 4), textcoords='offset points',
                fontsize=7.5, color='#2980B9')

# Regression line extended to ratio = 0
x_line = np.linspace(-0.02, bht['horner_ratio'].max() + 0.05, 100)
ax.plot(x_line, slope * x_line + intercept,
        color='#E74C3C', lw=1.5, ls='--', label='Horner extrapolation')

# Mark T_static at ratio = 0
ax.axvline(0, color='#555', lw=0.8, ls=':')
ax.scatter([0], [T_STATIC_HORNER], s=100, color='#E74C3C',
           zorder=6, marker='*', label=f'T_static = {T_STATIC_HORNER:.1f} °C')
ax.annotate(f'  T_static\n  {T_STATIC_HORNER:.1f} °C',
            xy=(0, T_STATIC_HORNER), xytext=(12, -2),
            textcoords='offset points', fontsize=8.5,
            color='#E74C3C', fontweight='bold')

ax.set_xlim(-0.05, bht['horner_ratio'].max() + 0.15)
ax.set_xlabel('Horner Time Ratio  =  log₁₀[(t_circ + Δt) / Δt]', fontsize=9)
ax.set_ylabel('Measured BHT (°C)', fontsize=9)
ax.set_title('Horner Correction — Representative BHT at ~3100 m TVDss\n'
             '(extrapolation to ratio = 0 → static formation temperature)',
             fontsize=9)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Annotate: infinite shut-in direction
ax.annotate('', xy=(-0.02, 68), xytext=(0.25, 68),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.2))
ax.text(0.12, 69.5, 'longer shut-in →', fontsize=7.5, color='gray', ha='center')

plt.tight_layout()
plt.show()
print(f"\nConclusion: static temperature at ~3100 m TVDss ≈ {T_STATIC_HORNER:.0f} °C")
print(f"This is consistent with a gradient of {GEOTHERM*1000:.0f} °C/km from seabed.")

## Step 2.3 — Geothermal Gradient & TEMP Log

With the Horner-corrected static temperature at depth and the known seabed temperature, we fit a linear geothermal gradient and apply it to every depth sample in the well.

The gradient is applied from the **seabed** (not from the surface or KB), since the water column above the seabed is at a near-uniform temperature and does not follow the geothermal gradient.

$$T(z) = T_{\text{seabed}} + G \times (z_{\text{TVDss}} - z_{\text{seabed}})$$

where $G$ = 0.031 °C/m and $z_{\text{seabed}}$ = 91.1 m below MSL.

In [ ]:
# ── TEMP log ──────────────────────────────────────────────────────────────────
# Water column: constant seabed temperature
# Below seabed: linear geothermal gradient
df['TEMP'] = T_SEABED + GEOTHERM * df['DEPTH_BELOW_SEABED']

# ── Helper: find nearest MD index ─────────────────────────────────────────────
def nearest_idx(df, md):
    return df.index[np.argmin(np.abs(df.index.values - md))]

# ── Verification against Horner result ────────────────────────────────────────
# Find temperature at ~3100 m TVDss
idx_3100 = (df['TVDSS_ABS'] - 3100).abs().idxmin()
T_at_3100 = df.loc[idx_3100, 'TEMP']
print(f"Gradient: {GEOTHERM*1000:.1f} °C/km from seabed")
print(f"T at seabed ({WATER_DEPTH:.0f} m TVDss)  : {T_SEABED:.1f} °C")
print(f"T at ~3100 m TVDss            : {T_at_3100:.1f} °C  (Horner result: {T_STATIC_HORNER:.1f} °C)")

# ── Key formation temperatures ────────────────────────────────────────────────
TOPS_MD = {
    'Ty Fm'       : 2621.5, 'Shetland GP' : 2770.6, 'Hod Fm'      : 2987.0,
    'Draupne Fm'  : 3358.0, 'Heather Fm'  : 3429.4, 'Hugin Fm'    : 3435.0,
    'Sleipner Fm' : 3500.2, 'Skagerrak Fm': 3543.7, 'Smith Bank Fm': 3608.0,
}
print(f"\n{'Formation':<20} {'MD (m)':>8} {'TVDss (m)':>10} {'T (°C)':>8}")
print("─" * 50)
for name, md_top in TOPS_MD.items():
    row = df.loc[nearest_idx(df, md_top)]
    print(f"{name:<20} {md_top:>8.1f} {row['TVDSS_ABS']:>10.1f} {row['TEMP']:>8.1f}")

## Step 2.4 — Formation Water Resistivity (Rw)

Formation water resistivity controls the Archie saturation calculation and the Pickett plot. It is a function of **salinity** and **temperature**.

**Salinity → Rw at 25°C** (Arps, 1953):
$$R_w^{25°C} = \left(\frac{3647.5}{\text{salinity [ppm]}}\right)^{0.955} + 0.0123$$

**Temperature correction** (Wright-Welex approximation):
$$R_w(T) = R_w^{25°C} \times \frac{25 + 21.5}{T + 21.5}$$

The Rw log is stored as a depth function (since temperature increases with depth).

In [ ]:
# ── Rw calculation ────────────────────────────────────────────────────────────
# Arps (1953): Rw@25°C from NaCl salinity in ppm
rw_25  = (3647.5 / SALINITY_PPM) ** 0.955 + 0.0123   # Rw at 25°C (Ω·m)
df['RW'] = rw_25 * (25 + 21.5) / (df['TEMP'] + 21.5)  # Rw at formation T

# Rw at key formations
print(f"Formation water salinity : {SALINITY_PPM:,} ppm NaCl")
print(f"Rw at 25°C               : {rw_25:.4f} Ω·m")
print()
print(f"{'Formation':<20} {'T (°C)':>8} {'Rw (Ω·m)':>10}")
print("─" * 42)
for name, md_top in TOPS_MD.items():
    row = df.loc[nearest_idx(df, md_top)]
    print(f"{name:<20} {row['TEMP']:>8.1f} {row['RW']:>10.4f}")

## Step 2.5 — Pressure Logs

Three pressure curves are needed for the rock physics model:

**Overburden (lithostatic) pressure** — integrate bulk density from the surface:
$$P_{\text{OB}}(z) = \int_0^z \rho_b(z') \cdot g \, dz'$$

Where $\rho_b$ comes from:
- 0 → seabed: seawater density (1.025 g/cc)
- Seabed → start of logging: compaction trend (no measured RHOB available)
- Logging interval: measured RHOB (nulls filled with compaction trend)

**Hydrostatic pore pressure** (normally pressured reservoir):
$$P_{\text{pore}}(z) = \rho_{\text{sw}} \cdot g \cdot z_{\text{TVDss}}$$

**Differential (effective) pressure** — the key input to the stress-sensitive RPM:
$$P_{\text{diff}} = P_{\text{OB}} - P_{\text{pore}}$$

In [ ]:
# ── Density model for pre-logging interval and null-filling ───────────────────
def rho_compaction_trend(tvdss_abs):
    """
    Bulk density compaction trend.
    Water column: seawater (1.025 g/cc).
    Sediment column: exponential approach from surface mudline density
    to a maximum consolidated density, calibrated to typical NNS values.
    """
    depth_in_rock = np.maximum(tvdss_abs - WATER_DEPTH, 0.0)
    rho = np.where(
        tvdss_abs <= WATER_DEPTH,
        RHO_SEAWATER,                                          # water column
        1.80 + 0.65 * (1 - np.exp(-depth_in_rock / 2000.0))   # compaction curve
    )
    return rho

# Check trend at a few depths
check_depths = np.array([0, 91, 500, 1000, 2000, 2600, 3100])
print("Compaction trend check:")
print(f"  {'TVDss (m)':>10}  {'Rho_trend (g/cc)':>18}")
for d in check_depths:
    print(f"  {d:>10.0f}  {rho_compaction_trend(d):>18.3f}")

In [ ]:
# ── Build overburden on a 1 m TVDss grid then interpolate to LAS depth ────────
z_max    = df['TVDSS_ABS'].max() + 10
z_grid   = np.arange(0.0, z_max + 1.0, 1.0)          # 1 m steps from MSL

# Density at each grid point — use measured RHOB within logging interval
rho_grid = rho_compaction_trend(z_grid).copy()

# Overwrite with measured RHOB where available
# Map LAS depth (MD) → TVDss on the 1m grid
rhob_valid = df['RHOB'].dropna()
if len(rhob_valid) > 0:
    tvdss_rhob = df.loc[rhob_valid.index, 'TVDSS_ABS'].values
    rhob_vals  = rhob_valid.values
    # Bin measured RHOB onto 1m grid (mean per bin)
    bins   = np.arange(0, z_max + 2, 1.0)
    idx    = np.digitize(tvdss_rhob, bins) - 1
    valid_idx = (idx >= 0) & (idx < len(rho_grid))
    for i, r in zip(idx[valid_idx], rhob_vals[valid_idx]):
        if not np.isnan(r):
            rho_grid[i] = r

# Integrate: cumulative trapezoidal sum × G_MPA (MPa per g/cc per m)
p_ob_grid = np.cumsum(rho_grid) * G_MPA    # dz = 1 m throughout

# ── Interpolate overburden back to LAS depth samples ─────────────────────────
df['P_OB']       = np.interp(df['TVDSS_ABS'], z_grid, p_ob_grid)
df['PORE_PRESS'] = df['TVDSS_ABS'] * RHO_SEAWATER * G_MPA   # hydrostatic
df['DIFF_PRESS'] = df['P_OB'] - df['PORE_PRESS']

# ── Summary at key horizons ────────────────────────────────────────────────────
print(f"Pressure summary (reservoir section):")
print(f"\n  {'Formation':<20} {'TVDss':>7} {'P_OB':>8} {'P_hydro':>9} {'P_diff':>8}")
print(f"  {'(m)':<20} {'(m)':>7} {'(MPa)':>8} {'(MPa)':>9} {'(MPa)':>8}")
print("  " + "─"*57)
for name, md_top in TOPS_MD.items():
    r = df.loc[nearest_idx(df, md_top)]
    print(f"  {name:<20} {r['TVDSS_ABS']:>7.0f} {r['P_OB']:>8.1f} "
          f"{r['PORE_PRESS']:>9.1f} {r['DIFF_PRESS']:>8.1f}")

## Step 2.6 — Composite T/P Profile Display

Three-track display of the computed curves vs TVDss depth, using the same inverted depth axis convention as the composite log display.

In [ ]:
# Restrict to the logging interval for display
LOG_TOP  = 2585   # MD
LOG_BASE = 3680   # MD
sub = df.loc[LOG_TOP:LOG_BASE].dropna(subset=['TVDSS_ABS'])
tvd = sub['TVDSS_ABS'].values

# Formation top positions (TVDss)
TOPS_TVDSS = {}
for name, md in TOPS_MD.items():
    TOPS_TVDSS[name] = df.loc[nearest_idx(df, md), 'TVDSS_ABS']
RESERVOIR_TOP = 'Hugin Fm'

fig, axes = plt.subplots(1, 3, figsize=(12, 11), sharey=True)
fig.subplots_adjust(top=0.93, bottom=0.07, left=0.10, right=0.97, wspace=0.10)

# ── Track 1 — Temperature ─────────────────────────────────────────────────────
ax = axes[0]
ax.plot(sub['TEMP'], tvd, color='#C0392B', lw=1.2)
# Shading between seabed temperature (cold) and curve (warm)
ax.fill_betweenx(tvd, T_SEABED, sub['TEMP'], alpha=0.15, color='#C0392B')
ax.set_xlim(0, 130)
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
ax.set_xlabel('Temperature (°C)', color='#C0392B', fontsize=9, labelpad=4)
ax.tick_params(axis='x', colors='#C0392B', labelsize=8)
ax.set_ylabel('Depth (m TVDss)', fontsize=10)
ax.set_title('TEMP', fontsize=10, fontweight='bold', pad=14)
ax.grid(True, alpha=0.25, lw=0.5)
ax.invert_yaxis()
ax.yaxis.set_major_locator(ticker.MultipleLocator(100))

# ── Track 2 — Rw ─────────────────────────────────────────────────────────────
ax = axes[1]
ax.plot(sub['RW'], tvd, color='#1A5276', lw=1.2)
ax.fill_betweenx(tvd, sub['RW'], sub['RW'].max(), alpha=0.15, color='#1A5276')
ax.set_xlim(0.04, 0.12)
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
ax.set_xlabel('Rw (Ω·m)', color='#1A5276', fontsize=9, labelpad=4)
ax.tick_params(axis='x', colors='#1A5276', labelsize=8)
ax.set_yticklabels([])
ax.set_title('Rw', fontsize=10, fontweight='bold', pad=14)
ax.grid(True, alpha=0.25, lw=0.5)

# ── Track 3 — Pressures ───────────────────────────────────────────────────────
ax = axes[2]
ax.plot(sub['P_OB'],       tvd, color='#884EA0', lw=1.2, label='P_overburden')
ax.plot(sub['PORE_PRESS'], tvd, color='#2980B9', lw=1.2, label='P_pore (hydrostatic)')
ax.plot(sub['DIFF_PRESS'], tvd, color='#117A65', lw=1.2, label='P_diff (effective)')
ax.set_xlim(0, 90)
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
ax.set_xlabel('Pressure (MPa)', fontsize=9, labelpad=4)
ax.tick_params(axis='x', labelsize=8)
ax.set_yticklabels([])
ax.legend(fontsize=7.5, loc='lower right', framealpha=0.8)
ax.set_title('PRESSURE', fontsize=10, fontweight='bold', pad=14)
ax.grid(True, alpha=0.25, lw=0.5)

# ── Formation tops across all tracks ─────────────────────────────────────────
from matplotlib.transforms import blended_transform_factory
label_trans = blended_transform_factory(axes[0].transAxes, axes[0].transData)

for name, tvdss_top in TOPS_TVDSS.items():
    if not (tvd.min() <= tvdss_top <= tvd.max()):
        continue
    is_res    = (name == RESERVOIR_TOP)
    lc        = '#1A5276' if is_res else '#5D4E37'
    for ax in axes:
        ax.axhline(tvdss_top, color=lc, lw=0.85, ls=(0, (7, 4)), alpha=0.85, zorder=4)
    axes[0].text(0.02, tvdss_top - (tvd.max()-tvd.min())*0.003, name,
                 transform=label_trans, fontsize=6.5, va='bottom', ha='left',
                 color=lc, fontweight='bold' if is_res else 'normal',
                 bbox=dict(facecolor='white', alpha=0.75, edgecolor='none', pad=1.2),
                 zorder=5)

fig.suptitle('15/9-F-1 A (Volve) — Temperature, Rw & Pressure Profiles',
             fontsize=11, fontweight='bold')
plt.show()

## Step 2.7 — Save Computed Curves

Save all computed curves (plus the MD→TVD conversion columns) to a Parquet file. Subsequent phase notebooks load this file alongside the LAS to access T/P logs without recomputing.

In [ ]:
# Select computed columns to persist
computed_cols = ['TVD', 'TVDSS_ABS', 'DEPTH_BELOW_SEABED',
                 'TEMP', 'RW', 'P_OB', 'PORE_PRESS', 'DIFF_PRESS']

df[computed_cols].to_parquet(OUT_FILE)

print(f"Saved to: {OUT_FILE}")
print(f"Columns : {computed_cols}")
print(f"Rows    : {len(df):,}  (depth index: MD, 0.1 m step)")
print()
print("Preview at Hugin Fm (MD 3435 m):")
idx = nearest_idx(df, 3435)
print(df.loc[idx, computed_cols].to_string())

---
## Phase 2 Summary

| Item | Value / Status |
|------|----------------|
| Geothermal gradient | 31 °C/km from seabed |
| T at Hugin Fm (3435 m MD / 3007 m TVDss) | ~98 °C |
| Formation water salinity | 57,500 ppm NaCl |
| Rw at 25°C (Arps formula) | ~0.082 Ω·m |
| Rw at Hugin Fm (~98 °C) | ~0.032 Ω·m |
| Pressure regime | Normally pressured |
| P_diff at Hugin Fm | ~37 MPa |
| Output saved | `wells/15_9-F-1A_computed.parquet` |

**Next**: Notebook `03_caliper_qc.ipynb` — hole condition QC and BAD_HOLE_FLAG (Phase 3).